# Interactive Multi-Agent Workflow - Sales Assist Tool

**Workflow:** Seller Query → Supervisory Agent → Contract Agent → Research Agent → Matching Agent → Action Agent → Results

## How to Use This Notebook

1. **Run Setup** - Install packages and initialize agents
2. **Step 1** - Ask your initial query
3. **Step 2** - Review and edit the draft email
4. **Step 3** - Confirm and send the email

---
## Setup and Environment Configuration

**Note**: The Contract Agent will automatically use cached data from `contracts_cache.json` if available!

---
## Step 0: Generate Contract Cache (First Time Only)

**IMPORTANT**: Run this cell ONCE to extract and cache all contract data.

**What it does**: Extracts text and structured fields (amount, products, dates) from contracts and saves to `contracts_cache.json`.

**Time**: ~1 minute (no LLM calls!)

**When to re-run**: Only when contract files change in `docs/` folder.

In [7]:
import os
import subprocess
import sys

# Check if cache exists
cache_exists = os.path.exists("contracts_cache.json")

if cache_exists:
    print("="*80)
    print("CONTRACT CACHE ALREADY EXISTS")
    print("="*80)
    print("Cache file found: contracts_cache.json")
    print("✓ Contracts will load instantly from cache")
    print("To regenerate cache (if contracts changed):")
    print("  1. Delete contracts_cache.json")
    print("  2. Run: python cache_contracts.py")
    print("="*80)
else:
    print("="*80)
    print("GENERATING CONTRACT CACHE")
    print("="*80)
    print("Extracting text and structured fields from contracts...")
    print("This only needs to be done ONCE.")
    
    try:
        result = subprocess.run(
            [sys.executable, "cache_contracts.py"],
            capture_output=True,
            text=True,
            timeout=300
        )
        
        if result.returncode == 0:
            print("✓ Cache generated successfully!")
            print("✓ Future runs will be 5-10x faster")
        else:
            print(f"Warning: {result.stderr}")
            print("Run manually: python cache_contracts.py")
    except Exception as e:
        print(f"Could not auto-generate: {e}")
        print("Run manually: python cache_contracts.py")
    
    print("="*80)

CONTRACT CACHE ALREADY EXISTS
Cache file found: contracts_cache.json
✓ Contracts will load instantly from cache
To regenerate cache (if contracts changed):
  1. Delete contracts_cache.json
  2. Run: python cache_contracts.py


In [2]:
# Install all requirements from requirements.txt
import sys
import subprocess

print("Installing packages from requirements.txt...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    print("All packages installed successfully!\n")
except subprocess.CalledProcessError as e:
    print(f"Error installing packages: {e}\n")
except FileNotFoundError:
    print("requirements.txt file not found!\n")

# Import required libraries
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Verify credentials
print("Environment Check:")
print(f"WATSONX_APIKEY: {'Set' if os.getenv('WATSONX_APIKEY') else 'Missing'}")
print(f"WATSONX_PROJECT_ID: {'Set' if os.getenv('WATSONX_PROJECT_ID') else 'Missing'}")
print(f"TAVILY_API_KEY: {'Set' if os.getenv('TAVILY_API_KEY') else 'Missing'}")

Installing packages from requirements.txt...
All packages installed successfully!

Environment Check:
WATSONX_APIKEY: Set
WATSONX_PROJECT_ID: Set
TAVILY_API_KEY: Set


## Initialize the Supervisory Agent

The Supervisory Agent orchestrates all other agents in the workflow.

In [8]:
from supervisory_agent import SupervisoryAgent

# Initialize the Supervisory Agent
print("Initializing Supervisory Agent...")
supervisor = SupervisoryAgent(
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID")
)
print("✓ Supervisory Agent ready\n")

Initializing Supervisory Agent...
Loaded scenario actions from docs/ScenarioActions.pdf (5143 characters)
✓ Supervisory Agent ready



---
## Interactive Workflow

### Step 1: Ask Your Initial Query

Enter your query below and run the cell to execute the full multi-agent workflow.

In [9]:
# Enter your query here
my_query = "I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps"

# You can also try these queries:
# my_query = "Can you give me an overview of all contracts and what I should do in the next 30 days?"
# my_query = "Which contracts are expiring soon and what are the next steps?"
# my_query = "I need to reach out to the CPO, can you draft me an email?"

print("="*80)
print("YOUR QUERY")
print("="*80)
print(f"\n{my_query}\n")
print("="*80)
print("Executing full workflow...")
print("="*80)

# Run the workflow
my_result = supervisor.run(
    seller_query=my_query,
    contract_file_path=None,
    partner_name="Confluent"
)

# Display results
print("\n" + "="*80)
print("WORKFLOW RESULTS")
print("="*80 + "\n")
print(my_result["final_result"])

YOUR QUERY

I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps

Executing full workflow...

SUPERVISORY AGENT - Workflow Initialization
Seller Query: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps

Intent Analysis:
NEXT_STEPS: [list of next steps based on the query]

INTENT: 
PARTNER_NAME: 
WORKFLOW_TYPE: 
CONTRACT_MENTIONED: 
KEY_ENTITIES: 
NEXT_STEPS: 

Please fill in the above format based on the analysis of the query.
INTENT: The seller wants to analyze contracts for renewal and expired contracts for Confluent.
PARTNER_NAME: Confl

### Step 2: Review and Edit the Draft Email

The initial workflow generated a draft email. You can now request edits or refinements to that email.

In [6]:
# First, let's extract the draft email from the initial results
initial_email = my_result.get("action_recommendation", {}).get("draft_email", "No email generated")

print("="*80)
print("ORIGINAL DRAFT EMAIL")
print("="*80)
print(f"\n{initial_email}\n")

# Now request an edit to the email
followup_query = "Can you make the email more urgent and add a specific deadline of April 15th for the response?"

# Other follow-up examples:
# followup_query = "Can you make the email shorter and more direct?"
# followup_query = "Can you add a mention of the $500K Cognos expansion opportunity?"
# followup_query = "Can you make the tone more friendly and less formal?"
# followup_query = "Can you add a bullet list of the key contracts we need to discuss?"

print("="*80)
print("EMAIL EDIT REQUEST")
print("="*80)
print(f"\n{followup_query}\n")
print("="*80)
print("Generating edited email...")
print("="*80)

# Use LLM to edit the email based on the request
from langchain_ibm import WatsonxLLM
from langchain_core.prompts import ChatPromptTemplate

llm = WatsonxLLM(
    model_id="meta-llama/llama-3-3-70b-instruct",
    url="https://us-south.ml.cloud.ibm.com",
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID"),
    params={
        "max_new_tokens": 600,
        "temperature": 0.3,
        "decoding_method": "sample"
    }
)

edit_prompt = ChatPromptTemplate.from_template(
    "You are helping edit a professional email. Here is the original email:\n\n"
    "{original_email}\n\n"
    "The user requests: {edit_request}\n\n"
    "Please provide a single edited version of the single email that incorporates this request. "
    "Maintain professional tone and include subject line. Do not repeat yourself and use proper grammar."
    "Please end at the end of the first IBM Seller signature"
)

formatted_prompt = edit_prompt.invoke({
    "original_email": initial_email,
    "edit_request": followup_query
})

edited_email = llm.invoke(formatted_prompt)
edited_email_text = edited_email.content if hasattr(edited_email, "content") else str(edited_email)

print("\n" + "="*80)
print("EDITED EMAIL")
print("="*80 + "\n")
print(edited_email_text)

ORIGINAL DRAFT EMAIL

Subject: Urgent: Confluent_IBM-1.30.2025 Contract Renewal

Dear Chief Procurement,

I am writing to bring to your attention the urgent need to renew our contract, Confluent_IBM-1.30.2025, which expired 67 days ago. With a value of $500,092.80, this renewal opportunity is at risk if not addressed promptly. Our discussions regarding the renewal are ongoing, and I would like to request an update on the current status.

As we move forward, it is crucial that we finalize the renewal to prevent any potential competitor entry during this gap period. I have been informed that the CFO has signed 1-year renewals, and I would appreciate any insight you can provide on the next steps.

I would appreciate the opportunity to discuss this further with you and explore ways to expedite the renewal process. I will be contacting Anand Das, the opportunity owner, to get a status update today.

Best regards,
John Doe
IBM Seller
```


Here is the rewritten response:


Subject:

EMAIL ED